In [24]:
# -------- Imports --------
import sys
import os
import numpy as np
import scipy.sparse as sp
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))
from _utility import *
from _fractures import *
from _initial_conditions import *


# -- Qiskit imports --
from qiskit import QuantumCircuit, QuantumRegister
from qiskit.circuit.library import PauliEvolutionGate
from qiskit.synthesis import MatrixExponential
from qiskit.quantum_info import SparsePauliOp, Pauli, Operator
from qiskit.primitives import StatevectorEstimator
from qiskit.quantum_info import Statevector

np.random.seed(0)

## Files Outputted from this Notebook

This file outputs dataframes corresponding to specific grid sizes, initial conditions, physical domains, etc. Listed below are the different physical scenarios we can try, feel free to update with more initial conditions or grid sizes, but do make sure to let the group know if you have added anything or you notice anything is wrong/different/interesting, etc. The grid sizes and physical domain can be changed in the _fractures.py file or moved to just be defined in this file separately.

### Just Classical

We have more freedom with calculating the subspace energies using the classical method, as it is not as limited by run time as the StatevectorEsimator is, so we can run the following scenarios

Physical Domain: Keep [-1,1] for x, y, and z dimensions for now

Grid Sizes: 6x6x6, 10x10x10, 19x19x19

Times: 20 times for each set, time ranges include [5e-6,1e-4], [5e-5,1e-3],[.005,.1],[.05,1],[.25,5],[.5,10]

### Quantum and Classical

Physical Domain: Keep [-1,1] for x,y, z dimensions for now

Grid Size: 6x6x6 (unless we find a way to run the StateVectorEstimator faster, this is the only one we can run in reasonable time

Times: 5 or so times is also the only reasonable amount of times to run this for, we can run for same time internvals [2e-5,1e-4], [2e-4,1e-3],[.02,.1],[.2,1],[1,5],[2,10]




### Define the grid parameters for the system
This should call the function get_grid_parameters() from _fractures.py script that contains all of the physical constants and information of the system. The script _fractures.py should be changed separately if a different system size or number of grid points is to be used. 

In [28]:
# -- Grid parameters --

# -- Lower/Upper Bound, Number of Points, Separation Between Points --

xmin,ymin,zmin,xmax,ymax,zmax,Nx,Ny,Nz,dx,dy,dz = get_grid_parameters()

# -- Staggered Grid Points for each wavefield component of seismic wave equation w = [v_x, v_y, v_z, σ_xx, σ_yy, σ_zz, σ_xy, σ_xz, σ_yz]

N_main,N_vx,N_vy,N_vz,N_sxy,N_sxz,N_syz,N_vel,N_stress,psi_len = get_grid_size(Nx,Ny,Nz)

In [29]:
## Configure which physical scenario to run for the notebook, times will be determined later
#Decide if quantum result will be calculated (takes a long time to run, don't run with over 6x6x6 case
calc_quantum = False


#If quantum is true, calculate quantum and classical, if false just calculate classical
file_label_1 = 'classical'
if calc_quantum:
    file_label_1='quantum'

#Decide which initial condition will be run, for now there is really only one, but we can edit when there are different ones, for now we will label the current one as ricker_IC_1
file_label_2 = 'ricker_IC_1'

#Use grid size to create this label 
file_label_3 = str(Nx)+'x'+str(Ny)+'x'+str(Nz)

#Use physical domain for this label

file_label_4 = str(xmin)+'to'+str(xmax)

#Fracture geometry 

file_label_5 = 'two_perp_xy_yz'
#file_label_5 = 'no_fracture'


## Define the indices of the mask that correspond with the subspace

This function returns the indices of the wave field vector that correspond with the subspace, or layer, of the system over which the expected value of the fraction of energy is calculated. It takes two arguments, "ith_element", and "wave_field_component,". "ith_element" tells the indices within the specified wave field component "subvector" corresponding with the subspace (for example, where in the vector v_x is included in the subspace rather than the whole vector itself).

In [33]:
#Define a function to get the indices of the points for the subspace energy calculation

def get_mask_indices(ith_element,wave_field_component):
    subspace_points = []
    for i in range(len(ith_element)):
        for j in ith_element[i]: 
            if(wave_field_component=='vx'):
                subspace_points.append(j)
            elif(wave_field_component=='vy'):
                subspace_points.append(N_vx+j)
            elif(wave_field_component=='vz'):
                subspace_points.append(N_vx+N_vy+j)
            elif(wave_field_component=='sxx'):
                subspace_points.append(N_vel+j)
            elif(wave_field_component=='syy'):
                subspace_points.append(N_vel+N_main+j)
            elif(wave_field_component=='szz'):
                subspace_points.append(N_vel+2*N_main+j)
            elif(wave_field_component=='sxy'): 
                subspace_points.append(N_vel+3*N_main+j)
            elif(wave_field_component=='sxz'):
                subspace_points.append(N_vel+3*N_main+N_sxy+j)
            elif(wave_field_component=='syz'):
                subspace_points.append(N_vel+3*N_main+N_sxy+N_sxz+j)
            else:
                print("Not a valid wave field component")
                
    return list(filter(lambda x: x < psi_len, subspace_points))

### Define a function for getting the "ith element" for each wave field component

Because each wave field component has different numbers of points in the wave field vector due to the staggered grid, there should be 7 different unique "i_locations" of the points for each "subvector." For example, while $v_x$ has $(Nx-1)*Ny*Nz$ points, $\sigma_{xy}$ has $(Nx-1)*(Ny-1)*Nz$ points, so there are different amounts of each of those points in the subspace that have to be indexed properly, or the mask will be off by a few index locations.

In [36]:
#Define a function to return the positions of points in the subspace within each component (i location)

def get_i_location(Nx,Ny,Nz,z_min,z_max,wave_field_component):
    
    assert z_min >= 0 and z_max <= Nz-1 and z_min < z_max
    i_location = []
    #Should be 7 different unique amounts of grid points for the same z location, the main grid, σ_xy, σ_xz, σ_yz, v_x, v_y, and v_z
    
    #vx
    if(wave_field_component=='vx'):
        i_location.append(np.arange((Nx-1)*Ny*z_min,(Nx-1)*Ny*z_max,1))
    #vy
    elif(wave_field_component=='vy'):
        i_location.append(np.arange(Nx*(Ny-1)*z_min,Nx*(Ny-1)*z_max,1))
    #vz
    elif(wave_field_component=='vz'):
        i_location.append(np.arange(Nx*Ny*z_min,Nx*Ny*(z_max),1))
    #main
    elif(wave_field_component=='sxx' or wave_field_component=='syy' or wave_field_component=='szz' ):
        i_location.append(np.arange(Nx*Ny*z_min,Nx*Ny*z_max,1))
    #stress_xy
    elif(wave_field_component=='sxy'):
        i_location.append(np.arange((Nx-1)*(Ny-1)*z_min,(Nx-1)*(Ny-1)*z_max,1))
    #stress_xz
    elif(wave_field_component=='sxz'):
        i_location.append(np.arange((Nx-1)*(Ny)*z_min,(Nx-1)*Ny*(z_max),1))
    #stress_yz
    elif(wave_field_component=='syz'):
        i_location.append(np.arange(Nx*(Ny-1)*z_min,Nx*(Ny-1)*(z_max),1))
    else:
        print('Not valid wave field component')

    #print(f'i_location {type(i_location)}: {i_location}')
    return i_location

### Define a function to get the subspaces of points for kinetic, potential and total energy

Each mask corresponds to points in the wave field. For kinetic energy, only velocities are used for the calculation so the other six entries of the wave field can be zeroed out. Vice versa for potential energy

In [38]:
# A function returning a dictionary of masks for kinetic, potential, and total energy subspace projectors
def threeProjectors(mask, Nx, Ny, Nz):
    kinetic = np.copy(mask)
    potential = np.copy(mask)
    kinetic_num = (Nx-1)*Ny*Nz + Nx*(Ny-1)*Nz + Nx*Ny*(Nz-1)
    kinetic[kinetic_num:] = 0
    potential[:kinetic_num] = 0
    return {
        'kinetic' : kinetic,
        'potential' : potential,
        'total' : mask
    }

### Define a function to get the expected value of the Hamiltonian simulation circuit

Currently uses StatevectorEsimator to find the expectation value of the quantum circuit defined later. This is the part of the code that takes a very long time to run if we increase the size of the system. Even if we increase the precision by many orders of magnitude, this function takes a long time to run and is not feasible to use for systems above 5x5x5 grid points. We should do more research to see if there is a way to decrease the run time of this operation with other qiskit functions.

In [41]:
# -------- Simulation --------
# Simulate the quantum circuit, calculate the expectation value of the energy in the subspace
def hamiltonian_simulation(qc,observable):
    estimator = StatevectorEstimator()
    job = estimator.run([(qc, observable)], precision=1e-8)
    result = job.result()
    print(f" Layer > Expectation value: {result[0].data.evs}")
    print(f" > Metadata: {result[0].metadata}")
    return result

### Material parameters and definition of the Hamiltonian

We must determine how defining the material parameters on a quantum computer will be done. In the code, it mentions this code "assumes oracle access," so __it might be a good idea here to determine the complexity of these oracles and see how everything scales__.

Here is where the fracture density and compliance matrix are defined to define the $B$ operator, $A$ operator, and thus the Hamiltonian. This calls functions in the _utility.py script as well as the _fractures.py script. It also converts the Hamiltonian to a string of Pauli operators, which is noted as an expensive coversion. __It might be worth looking into how we could construct a more efficient circuit without using H_pauli and direct matrix exponentiation__

In [44]:
# Important: The following implementation of the material parameters is not scalable. This code is only for demonstration purposes and assumes oracle access. 
# In a real-world scenario, the material parameters and the Hamiltonian need to be sparsly constructed on the QC itself.
# Furthermore, this code uses direct matrix exponentiation, which is inefficient for large matrices. Other integration methods should be used (e.g. Trotter-Suzuki, Qubitization).


# Density model and S matrix (3D with one horizontal fracture)

#Set fracture thickness
fracture_thickness = 0
#Set density and compliance matrix according to fracture geometry specified earlier
if file_label_5 == 'two_perp_xy_yz':
    rho_model, S_matrix = two_perpendicular_fractures(Nx,Ny,Nz,dx,dy,dz,fracture_thickness)
elif file_label_5 == 'no_fracture':
    rho_model, S_matrix = no_fractures(Nx,Ny,Nz,dx,dy,dz)
elif file_label_5 == 'one_xy':
    rho_model, S_matrix = one_horizontal_fracture(Nx,Ny,Nz,dx,dy,dz,fracture_thickness)
elif file_label_5 == 'one_yz':
    rho_model, S_matrix = one_vertical_fracture(Nx,Ny,Nz,dx,dy,dz,fracture_thickness)
else:
    rho_model, S_matrix = no_fractures(Nx,Ny,Nz,dx,dy,dz)

# -------- Simulation (3D Elastic) (Oracle Access) --------
(H, A, _, B_sqrt, B_inv, _) = FD_solver_3D_elastic_quantum(Nx, Ny, Nz, dx, dy, dz, rho_model, S_matrix)

H_pauli = SparsePauliOp.from_operator(Operator(H.toarray())) # Expensive conversion
print("Hamiltonian shape: ", H.shape)
print("Hamiltonian NNZ-Ratio: ", H.nnz/H.shape[0]**2)
print("Pauli Terms (inefficient representation): ", len(H_pauli))

# Number of grid points
print("Number of grid points: ", psi_len)

# Number of qubits
n = (H.shape[0]-1).bit_length()
print("Number of qubits (for wave field): ", n)


Adding perpendicular fracture planes:
  Fracture: Vertical plane at x-index 4
  Fracture: Horizontal plane at z-index 4
  Total fracture cells: 45 out of 1000 (4.5%)

Density Model Statistics:
  Min: 1000.0 kg/m³, Max: 2700.0 kg/m³
  Mean: 2623.5 kg/m³, Std: 352.4 kg/m³
Hamiltonian shape:  (8192, 8192)
Hamiltonian NNZ-Ratio:  0.0004506111145019531
Pauli Terms (inefficient representation):  4407216
Number of grid points:  8130
Number of qubits (for wave field):  13


### Define initial condition for the wave field here, and then transform it to the energy basis and normalize it for Hamiltonian simulation

Initial conditions for the wave field can give non zero velocity and nonzero stress. The gaussian_IC gives non zero normal stress, while the ricker_IC gives nonzero velocity in the x-direction. __We should expand the initial conditions we use to include more physical scenarios to see how that changes the results. More research into physically relevant seismic initial conditions should be done.__

In [47]:
# -- Initial Conditions (3D Elastic) --
# State vector: [v_x, v_y, v_z, σ_xx, σ_yy, σ_zz, σ_xy, σ_xz, σ_yz]

#phi_0 = gaussian_IC(Nx,Ny,Nz,dx,dy,dz,xmin,xmax,ymin,ymax,zmin,zmax)
if file_label_2 == 'ricker_IC_1':
    phi_0 = ricker_IC(Nx,Ny,Nz,dx,dy,dz,xmin,xmax,ymin,ymax,zmin,zmax)
elif file_label_2 == 'gaussian_IC_1':
    phi_0 = gaussian_IC(Nx,Ny,Nz,dx,dy,dz,xmin,xmax,ymin,ymax,zmin,zmax)
else:
    phi_0 = ricker_IC(Nx,Ny,Nz,dx,dy,dz,xmin,xmax,ymin,ymax,zmin,zmax)
    
# Pad the initial conditions with zeros to match Hamiltonian size (power of 2)
phi_0 = np.concatenate([phi_0, np.zeros(H.shape[0] - psi_len)])
# Normalize the initial state and transform it to the energy basis
psi_0 = B_sqrt @ phi_0
norm = np.linalg.norm(psi_0)
psi_0 /= norm

# Number of non-zero initial state values
psi_0_nnz = np.sum(psi_0 != 0)
print('Initial state NNZ-Ratio:', psi_0_nnz/psi_len)

Initial state NNZ-Ratio: 0.06691266912669126


### Define the quantum circuit for Hamiltonian simulation (the function takes a mask for the subspace being calculated and a time up to which the state should be evolved

The quantum circuit is initialized with $n$ qubits, where $n$ depends on the size of the Hamiltonian, which is a $2^nx2^n$ matrix. There is one register of $n$ qubits for the wave field, and one ancillary qubit register for measurement.

The quantum circuit prepares the state with the normalized wave field vector defined earlier, the time evolution operator is constructed and appedended to the circuit, and a unitary projection transform is applied to make sure the subspace is being measured (this changes all ancillary qubits to state 0 that are in the subspace instead of state 1 which they are initialized to). Then, the observable is defined so the wave field can be measured using the ancillary qubit register. __Complexity of running this circuit must be determined__.

In [49]:
def get_quantum_circuit(mask,t):
    synthesis = MatrixExponential()
    evo = PauliEvolutionGate(H_pauli, time=t, synthesis=synthesis, label='U=exp(-iH_real*t)')

    # -- Quantum Simulation --
    # Define the quantum registers
    reg_wave = QuantumRegister(n, 'wave')
    reg_P = QuantumRegister(1, 'P')

    # Initialize the quantum circuit
    qc = QuantumCircuit(reg_wave, reg_P)
    qc.prepare_state(psi_0, reg_wave)
    print("Normalization: ",np.linalg.norm(psi_0))
    # Apply the time-evolution operator
    qc.append(evo, reg_wave)

    # Apply unitary "projection" transform (Can often be combined into fewer MCX gates)
    qc.x(reg_P)
    for d in np.nonzero(mask)[0]:
        ctrl_bin = format(d, f'0{n}b')  # binary string of length n
        qc.mcx(reg_wave, target_qubit=reg_P, ctrl_state=ctrl_bin)

    # Define observable
    O1 = Pauli('I'*(n+1))
    O2 = Pauli('Z'+'I'*n) 
    observable = SparsePauliOp([O1, O2], [0.5, 0.5])

    return qc,observable

### Functions for getting subspace energy

Below are functions for getting the subspace energy, calculated classically and using the quantum circuit. The quantum subspace energy is calculated by running the circuit, finding the expectation value E of the fraction of energy distributed in the subspace, or layer, and then multiplying it by 1/2 times the norm of the wave field times the volume of the subspace. The volume of the subspace should involved dx, dy, and dz to scale the energy density to have units of energy.

In [51]:
# Compute the energy loss (post-processing)
def get_quantum_subspace_energy(result):
    E = np.abs(result[0].data.evs)
    EN_QC = (1/2) * E * norm**2 * (dx * dy * dz)
    print('Expected value: ',E)
    print('Subspace Energy Estimate (quantum):', EN_QC.round(30))
    return EN_QC.round(30)

In [52]:
# Time Integration (Has classical integration errors (DOP853))
def get_classical_state_vector(t):
    #phi_t = solve_ivp(lambda t, phi: (B_inv @ A @ phi), (0, t), phi_0, t_eval=(0,t), method='DOP853').y.T[-1] 
    phi_t = solve_ivp(lambda t, phi: (B_inv @ A @ phi), (0, t), phi_0, t_eval=(0,t), method='RK45').y.T[-1] 
    psi_t_classical = B_sqrt @ phi_t
    return psi_t_classical

In [53]:
def get_classical_subspace_energy_1(mask,psi_t_classical):
    masked_state = np.diag(mask) @ psi_t_classical
    
    # Probability of being in the masked subspace
    prob_in_subspace = np.linalg.norm(masked_state)**2 / np.linalg.norm(psi_t_classical)**2
    
    print("prob in subspace",prob_in_subspace)
    
    EN_CL = (1/2) * prob_in_subspace * np.linalg.norm(psi_t_classical)**2 * (dx * dy * dz)
    print('Subspace Energy Estimate (classical 1):', EN_CL.round(30))
    return EN_CL.round(30)

In [24]:
def get_classical_subspace_energy(mask,psi_t_classical):

    EN_CL = (1/2) * np.linalg.norm(np.diag(mask) @ psi_t_classical, axis=0)**2 * (dx * dy * dz)
    print('Subspace Energy Estimate (classical):', EN_CL.round(30))
    return EN_CL.round(30)

### Test one layer to make sure the code is working

Next few cells test one layer to see how long each subspace will take to run for each time

In [37]:
## Test to see if the code is working by just defining one mask and one time 

#Define some layer to get the subspace for

layer_min = 3
layer_max = 4

test_mask = (subspaceProjector(psi_len,get_mask_indices(get_i_location(Nx,Ny,Nz,layer_min,layer_max,'vx'),'vx'))
+subspaceProjector(psi_len,get_mask_indices(get_i_location(Nx,Ny,Nz,layer_min,layer_max,'vy'),'vy'))
+subspaceProjector(psi_len,get_mask_indices(get_i_location(Nx,Ny,Nz,layer_min,layer_max,'vz'),'vz'))
+subspaceProjector(psi_len,get_mask_indices(get_i_location(Nx,Ny,Nz,layer_min,layer_max,'sxx'),'sxx'))
+subspaceProjector(psi_len,get_mask_indices(get_i_location(Nx,Ny,Nz,layer_min,layer_max,'syy'),'syy'))
+subspaceProjector(psi_len,get_mask_indices(get_i_location(Nx,Ny,Nz,layer_min,layer_max,'szz'),'szz'))
+subspaceProjector(psi_len,get_mask_indices(get_i_location(Nx,Ny,Nz,layer_min,layer_max,'sxy'),'sxy'))
+subspaceProjector(psi_len,get_mask_indices(get_i_location(Nx,Ny,Nz,layer_min,layer_max,'sxz'),'sxz'))
+subspaceProjector(psi_len,get_mask_indices(get_i_location(Nx,Ny,Nz,layer_min,layer_max,'syz'),'syz')))
test_mask = np.concatenate([test_mask, np.zeros(H.shape[0]-psi_len)])
test_time = 1e-4



In [31]:
## Get the test qc and observable 

qc,observable = get_quantum_circuit(test_mask,test_time)

Normalization:  1.0


In [ ]:
## Get the test result 

result = hamiltonian_simulation(qc,observable)

In [234]:
## Get the test quantum subspace energy

qc_sub_energy = get_quantum_subspace_energy(result)

Expected value:  8.22675261247241e-14
Subspace Energy Estimate (quantum): 9.024430446894262e-05


In [235]:
## Get the classical state vector

psi_t_classical = get_classical_state_vector(test_time)

In [236]:
## Get two different estimations of classical subspace energy to make sure they equal each other

cl_sub_energy_1 = get_classical_subspace_energy_1(test_mask,psi_t_classical)

cl_sub_energy_2 = get_classical_subspace_energy(test_mask,psi_t_classical)


prob in subspace 2.2359116831068835e-15
Subspace Energy Estimate (classical 1): 2.452708914451286e-06
Subspace Energy Estimate (classical 2): 2.4527089144512866e-06


In [26]:
## RUN THE SIMULATION FOR MULTIPLE LAYERS AND TIMES (start with eight layers, can change if we decide to do fewer or more)
num_layers = 5 #9, change to 5 if we are running 6x6x6 case
num_times = 5 #5, change to more times for classical only simulations
t = 1
layer_spacing = (Nz-1)//num_layers
times = np.linspace(t/num_times,t,num_times)
layer_min = np.arange(0,Nz-1,layer_spacing)
layer_max = layer_min+layer_spacing

## MASK DEFINITION FOR TOTAL, KINETIC, and POTENTIAL ENERGIES

masks = []
mask_layer = []
components = ['vx','vy','vz','sxx','syy','szz','sxy','sxz','syz']
for i in range(num_layers):
    mask_layer=[]
    mask_kinetic = np.concatenate([np.zeros(psi_len),np.zeros(H.shape[0]-psi_len)])
    mask_potential = np.concatenate([np.zeros(psi_len),np.zeros(H.shape[0]-psi_len)])
    mask_total = np.concatenate([np.zeros(psi_len),np.zeros(H.shape[0]-psi_len)])
    for j in range(len(components)):
        mask = subspaceProjector(psi_len,get_mask_indices(get_i_location(Nx,Ny,Nz,layer_min[i],layer_max[i],components[j]),components[j]))
        mask = np.concatenate([mask, np.zeros(H.shape[0]-psi_len)])
        mask_total+=mask
        if(components[j]=='vx' or components[j]=='vy' or components[j]=='vz'):
            #print("kinetic")
            mask_kinetic+=mask
        else:
            #print("elastic")
            mask_potential+=mask
    mask_layer.append(mask_kinetic)
    mask_layer.append(mask_potential)
    mask_layer.append(mask_total)
    masks.append(mask_layer)

In [27]:
## RUN THE SIMULATION FOR MULTIPLE TIMES, choose whether to include both quantum and classical or just classical energies
## Just classical will allow for a faster run time and more grid points, quantum is limited to 5x5x5 case for now due to the long run time
    

subspace_energies_total_quantum = []
subspace_energies_kinetic_quantum = []
subspace_energies_elastic_quantum = []
subspace_energies_total_classical = []
subspace_energies_kinetic_classical = []
subspace_energies_elastic_classical = []

time_subspace_energies_total_quantum = []
time_subspace_energies_kinetic_quantum = []
time_subspace_energies_elastic_quantum = []
time_subspace_energies_total_classical = []
time_subspace_energies_kinetic_classical = []
time_subspace_energies_elastic_classical = []

energy_type = ['kinetic','potential','total']

test_mask = masks[0] #Contains 9 masks for each component
qe_layer = 0
ce_layer = 0
for time in range(len(times)):
    #Only have to calculate classical state vector once per time
    psi_t_classical = get_classical_state_vector(times[time])
    subspace_energies_total_quantum = []
    subspace_energies_kinetic_quantum = []
    subspace_energies_elastic_quantum = []
    subspace_energies_total_classical = []
    subspace_energies_kinetic_classical = []
    subspace_energies_elastic_classical = []
    for j in range(len(masks)):
        qe_layer=0
        ce_layer=0
        qe_layer_k=0
        ce_layer_k=0
        qe_layer_e=0
        ce_layer_e=0
        layer_mask = masks[j]
        for k in range(len(layer_mask)):
            qe_mask = 0
            ce_mask = 0
            if calc_quantum:
                qc, observable = get_quantum_circuit(layer_mask[k],times[time])
                result = hamiltonian_simulation(qc,observable)
                qe_mask = get_quantum_subspace_energy(result)
            ce_mask = get_classical_subspace_energy(layer_mask[k],psi_t_classical)
            if(energy_type[k]=='kinetic'):
                qe_layer_k+=qe_mask
                ce_layer_k+=ce_mask
            elif(energy_type[k]=='potential'):
                qe_layer_e+=qe_mask
                ce_layer_e+=ce_mask
            else:
                qe_layer+=qe_mask
                ce_layer+=ce_mask
        print("Quantum Energy Layer: ",j," Energy Type (cumulative): ",energy_type[k]," ",qe_layer)
        print("Classical Energy Layer: ",j,"Energy Type (cumulative): ",energy_type[k]," ",ce_layer)

        subspace_energies_total_quantum.append(qe_layer)
        subspace_energies_total_classical.append(ce_layer)
        subspace_energies_kinetic_quantum.append(qe_layer_k)
        subspace_energies_kinetic_classical.append(ce_layer_k)
        subspace_energies_elastic_quantum.append(qe_layer_e)
        subspace_energies_elastic_classical.append(ce_layer_e)

    time_subspace_energies_total_quantum.append(subspace_energies_total_quantum)
    time_subspace_energies_total_classical.append(subspace_energies_total_classical)
    time_subspace_energies_kinetic_quantum.append(subspace_energies_kinetic_quantum)
    time_subspace_energies_kinetic_classical.append(subspace_energies_kinetic_classical)
    time_subspace_energies_elastic_quantum.append(subspace_energies_elastic_quantum)
    time_subspace_energies_elastic_classical.append(subspace_energies_elastic_classical)

Subspace Energy Estimate (classical): 1.4250828197317524e-14
Subspace Energy Estimate (classical): 3.015447093387121e-12
Subspace Energy Estimate (classical): 3.0296979215844387e-12
Quantum Energy Layer:  0  Energy Type (cumulative):  total   0
Classical Energy Layer:  0 Energy Type (cumulative):  total   3.0296979215844387e-12
Subspace Energy Estimate (classical): 8.311573991172598e-09
Subspace Energy Estimate (classical): 2.6765640001119237e-06
Subspace Energy Estimate (classical): 2.6848755741030966e-06
Quantum Energy Layer:  1  Energy Type (cumulative):  total   0
Classical Energy Layer:  1 Energy Type (cumulative):  total   2.6848755741030966e-06
Subspace Energy Estimate (classical): 0.000578220698563712
Subspace Energy Estimate (classical): 0.02445018401949725
Subspace Energy Estimate (classical): 0.02502840471806095
Quantum Energy Layer:  2  Energy Type (cumulative):  total   0
Classical Energy Layer:  2 Energy Type (cumulative):  total   0.02502840471806095
Subspace Energy Esti

In [28]:
time_subspace_energies_total_quantum = np.array(time_subspace_energies_total_quantum)
time_subspace_energies_total_classical = np.array(time_subspace_energies_total_classical)
time_subspace_energies_kinetic_quantum = np.array(time_subspace_energies_kinetic_quantum)
time_subspace_energies_kinetic_classical = np.array(time_subspace_energies_kinetic_classical)
time_subspace_energies_elastic_quantum = np.array(time_subspace_energies_elastic_quantum)
time_subspace_energies_elastic_classical = np.array(time_subspace_energies_elastic_classical)


In [30]:

import pandas as pd
import numpy as np
time_array=[]
for i in range(len(times)):
    for j in range(len(subspace_energies_total_quantum)):
        time_array.append(times[i])
        
total_quantum=time_subspace_energies_total_quantum.flatten()
total_classical=time_subspace_energies_total_classical.flatten()
kinetic_quantum=time_subspace_energies_kinetic_quantum.flatten() 
kinetic_classical=time_subspace_energies_kinetic_classical.flatten()
potential_quantum=time_subspace_energies_elastic_quantum.flatten()
potential_classical=time_subspace_energies_elastic_classical.flatten()

## CREATE PANDAS DATAFRAME TO EXPORT AS A CSV
data = {
    'total_quantum':total_quantum,
    'total_classical': total_classical,
    'kinetic_quantum': kinetic_quantum,
    'kinetic_classical': kinetic_classical,
    'potential_quantum': potential_quantum,
    'potential_classical': potential_classical,
    'time':time_array
}
df = pd.DataFrame(data)

In [32]:
## ADD A COLUMN LABELING THE LAYERS

df['Layer #'] = np.ones(len(data['total_quantum']),dtype=int)
index_counter = 0
for i in range(df['time'].nunique()):
    layer_counter = 0
    for j in range(num_layers):
        df.loc[index_counter,'Layer #']=layer_counter+1
        layer_counter+=1
        index_counter+=1

In [33]:
df

,total_quantum,total_classical,kinetic_quantum,kinetic_classical,potential_quantum,potential_classical,time,Layer #
0,0,3.029698e-12,0,1.425083e-14,0,3.015447e-12,0.000005,1
1,0,2.684876e-06,0,8.311574e-09,0,2.676564e-06,0.000005,2
2,0,2.502840e-02,0,5.782207e-04,0,2.445018e-02,0.000005,3
3,0,4.014431e+00,0,3.927755e+00,0,8.667560e-02,0.000005,4
4,0,4.091892e+00,0,3.982847e+00,0,1.090448e-01,0.000005,5
...,...,...,...,...,...,...,...,...
175,0,4.167610e+00,0,2.044328e+00,0,2.123282e+00,0.000100,5
176,0,3.571251e+00,0,1.948795e+00,0,1.622456e+00,0.000100,6
177,0,3.076216e+00,0,1.354141e+00,0,1.722075e+00,0.000100,7
178,0,9.088990e-01,0,4.672409e-01,0,4.416580e-01,0.000100,8


In [34]:
#CREATE FILE NAME THAT IS DESCRIPTIVE FOR THE DATA IT HOLDS
#As follows (if labeled quantum, includes classical and quantum values
#energies_classicalorquantum_initialcondition_gridsize_domain_times

#FOR EXAMPLE
#energies_classical_ricker_IC_1_9x9x9_-1to1_2e-5_1e-4
file_name = 'energies_'+ file_label_1+ '_'+file_label_2+'_'+file_label_3+'_'+file_label_4+'_'+file_label_5+'_'+str(times[0])+'_'+str(times[num_times-1])
file_name

'energies_classical_ricker_IC_1_10x10x10_-1to1_no_fracture_5e-06_0.0001'

In [53]:
#Convert to a csv file with a descriptive file name
df.to_csv(file_name, index=False)